<a href="https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/20260825/notebooks/qsar_practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QSAR & ADMET 실습 — SMILES → 기술자 → 물성·약물성 예측

**AI 신약개발 · 화합물 표현 & QSAR (3교시 연계 실습)**

분자를 **기술자/지문으로 표현**하고, 그 표현으로 **회귀(QSAR)** 와 **분류(ADMET)** 를 모두 예측합니다 — 같은 "기술자→모델" 기계가 두 문제에 그대로 쓰입니다.

1. **데이터** — ESOL(Delaney) 수용해도 (회귀, 1128 분자)
2. **표현(2교시 연계)** — RDKit **분자 기술자**(MW·logP·TPSA…) vs **Morgan/ECFP 지문**
3. **모델(QSAR 회귀)** — 선형회귀·랜덤포레스트로 logS 예측
4. **검증(3교시 핵심)** — 교차검증 · **y-scrambling** · **적용범위(AD)**
5. **시각화** — 예측-실측 산점도 · 잔차 · 특징 중요도
6. **ADMET 확장(분류)** — 혈뇌장벽 투과(**BBBP**) 예측: ROC-AUC·혼동행렬

> ⚠️ **무-날조**: 데이터는 실제 측정/주석값, 모든 지표(R²·RMSE·ROC-AUC 등)는 이 노트북이 **실제로 계산**한 값입니다.
> 🖥️ CPU로 수 분 내 완료. RDKit·scikit-learn만 필요.

## 0. 설치 & 환경

In [ ]:
!pip install -q rdkit scikit-learn
# 한글 폰트(그래프 깨짐 방지)
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true
import matplotlib as mpl, matplotlib.font_manager as fm
_kf=[f for f in fm.findSystemFonts() if "Nanum" in f]
for _f in _kf: fm.fontManager.addfont(_f)
if _kf: mpl.rcParams["font.family"]="NanumGothic"
mpl.rcParams["axes.unicode_minus"]=False
import platform; print("python", platform.python_version(), "| 한글폰트:", "OK" if _kf else "기본")


## 1. 데이터 — ESOL 수용해도

Delaney(2004)의 **측정 수용해도**(log mol/L) 데이터. 각 분자는 SMILES로 주어집니다.

In [ ]:
import pandas as pd, numpy as np
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog("rdApp.*")

URL="https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv"
df=pd.read_csv(URL)
df=df.rename(columns={"measured log solubility in mols per litre":"logS"})[["smiles","logS"]]
df["mol"]=df["smiles"].apply(Chem.MolFromSmiles)
df=df[df["mol"].notnull()].reset_index(drop=True)   # RDKit로 파싱되는 분자만
print("분자 수:", len(df), "| logS 범위: %.2f ~ %.2f" % (df.logS.min(), df.logS.max()))
df[["smiles","logS"]].head()


## 2. 표현 (2교시 연계) — 기술자 vs 지문

- **분자 기술자**: 물성·위상 기반 해석 가능한 소수 차원(MW·logP·TPSA·HBD/HBA·회전결합·방향족고리 등)
- **Morgan/ECFP 지문**: 원형 부분구조를 비트벡터로(고차원, 강력)

In [ ]:
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator

def descriptors(m):
    return [
        Descriptors.MolWt(m), Crippen.MolLogP(m), rdMolDescriptors.CalcTPSA(m),
        Lipinski.NumHDonors(m), Lipinski.NumHAcceptors(m),
        rdMolDescriptors.CalcNumRotatableBonds(m), rdMolDescriptors.CalcNumAromaticRings(m),
        rdMolDescriptors.CalcNumRings(m), Descriptors.FractionCSP3(m), m.GetNumHeavyAtoms(),
    ]
DESC_NAMES=["MW","logP","TPSA","HBD","HBA","RotB","AromRings","Rings","FracCSP3","HeavyAtoms"]

X_desc=np.array([descriptors(m) for m in df["mol"]])
mfp=rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)   # ECFP4 상당
X_fp=np.array([list(mfp.GetFingerprint(m)) for m in df["mol"]])
y=df["logS"].values
print("기술자 행렬:", X_desc.shape, "| ECFP 지문 행렬:", X_fp.shape)


## 3. QSAR 모델 — logS 예측

표현(기술자·지문) × 모델(선형회귀·랜덤포레스트)을 **동일 train/test 분할**에서 공정 비교.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error

Xtr_d,Xte_d,ytr,yte=train_test_split(X_desc,y,test_size=0.2,random_state=42)
Xtr_f,Xte_f,_,_   =train_test_split(X_fp,  y,test_size=0.2,random_state=42)

def evaluate(Xtr,Xte,model):
    model.fit(Xtr,ytr); p=model.predict(Xte)
    return r2_score(yte,p), mean_squared_error(yte,p)**0.5, p

runs={
 "기술자 + Ridge":     evaluate(Xtr_d,Xte_d, make_pipeline(StandardScaler(),Ridge(alpha=1.0))),
 "기술자 + RandomForest": evaluate(Xtr_d,Xte_d, RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)),
 "ECFP + RandomForest":  evaluate(Xtr_f,Xte_f, RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)),
}
res=pd.DataFrame([{"표현+모델":k,"R²":round(v[0],3),"RMSE":round(v[1],3)} for k,v in runs.items()])
print("=== 테스트셋 성능 (실측) ==="); res


## 4. 검증 (3교시 핵심) — 신뢰할 수 있는 QSAR인가?

- **교차검증(CV)**: 분할 우연성 배제
- **y-scrambling**: 라벨을 섞으면 성능이 **무너져야** 정상(우연 상관이 아님을 증명)
- **적용범위(Applicability Domain)**: train 분포에서 먼 분자는 예측 신뢰도↓

In [ ]:
from sklearn.model_selection import cross_val_score
rf=RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)

cv=cross_val_score(rf,X_desc,y,cv=5,scoring="r2")
print(f"5-fold CV R² (기술자+RF): {cv.mean():.3f} ± {cv.std():.3f}")

# y-scrambling: 라벨을 섞어 학습 → 성능 붕괴 확인
rng=np.random.RandomState(0)
scr=[cross_val_score(rf, X_desc, rng.permutation(y), cv=5, scoring="r2").mean() for _ in range(10)]
print(f"y-scramble CV R² (평균 {np.mean(scr):.3f}) → 실제 모델({cv.mean():.3f})보다 크게 낮아야 정상")


In [ ]:
# 적용범위(AD): train 표준화 기술자 공간에서의 거리(k-NN 평균거리)로 근접도 평가
from sklearn.neighbors import NearestNeighbors
sc=StandardScaler().fit(Xtr_d)
nn=NearestNeighbors(n_neighbors=5).fit(sc.transform(Xtr_d))
d_test,_=nn.kneighbors(sc.transform(Xte_d))
ad_score=d_test.mean(axis=1)                     # 작을수록 train에 가까움(AD 내부)
thr=np.percentile(nn.kneighbors(sc.transform(Xtr_d))[0].mean(axis=1),95)  # train 95퍼센타일
inside=ad_score<=thr
_,rmse_rf,p_rf=runs["기술자 + RandomForest"]
from sklearn.metrics import mean_squared_error
print(f"AD 내부 {inside.sum()}/{len(inside)} 분자")
print(f"AD 내부 RMSE={mean_squared_error(yte[inside],p_rf[inside])**0.5:.3f} | "
      f"AD 외부 RMSE={mean_squared_error(yte[~inside],p_rf[~inside])**0.5:.3f}  (외부가 대체로 더 큼)")


## 5. 시각화

In [ ]:
import matplotlib.pyplot as plt

fig,axes=plt.subplots(1,3,figsize=(16,4.6))
# (1) 표현+모델 성능 비교 막대
axes[0].bar(res["표현+모델"], res["R²"], color=["#4C72B0","#55A868","#C44E52"])
axes[0].set_ylabel("테스트 R²"); axes[0].set_title("표현×모델 성능"); axes[0].tick_params(axis="x",rotation=20)
for i,v in enumerate(res["R²"]): axes[0].text(i,v+0.01,f"{v}",ha="center",fontsize=9)
# (2) 예측 vs 실측 (기술자+RF)
axes[1].scatter(yte,p_rf,s=14,alpha=0.5,edgecolor="k",linewidth=0.3,c="#4C72B0")
lim=[min(yte.min(),p_rf.min()),max(yte.max(),p_rf.max())]
axes[1].plot(lim,lim,"r--",lw=1)
r2v,rmsev,_=runs["기술자 + RandomForest"]
axes[1].set_xlabel("실측 logS"); axes[1].set_ylabel("예측 logS")
axes[1].set_title(f"예측 vs 실측 (기술자+RF)\nR²={r2v:.3f}  RMSE={rmsev:.3f}")
# (3) 잔차
axes[2].scatter(p_rf, yte-p_rf, s=14, alpha=0.5, c="#55A868", edgecolor="k", linewidth=0.3)
axes[2].axhline(0,color="r",ls="--",lw=1); axes[2].set_xlabel("예측 logS"); axes[2].set_ylabel("잔차")
axes[2].set_title("잔차 플롯")
plt.tight_layout(); plt.show()


In [ ]:
# 특징 중요도 (기술자+RF) — 어떤 물성이 용해도를 좌우하나
rf_d=RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1).fit(Xtr_d,ytr)
imp=pd.Series(rf_d.feature_importances_, index=DESC_NAMES).sort_values()
plt.figure(figsize=(7,4.5))
plt.barh(imp.index, imp.values, color="#4C72B0")
plt.xlabel("중요도(랜덤포레스트)"); plt.title("기술자 중요도 — logS 예측")
plt.tight_layout(); plt.show()
print("상위 3:", list(imp.sort_values(ascending=False).index[:3]))


## 6. (확장) ADMET 예측 — 분류 문제: 혈뇌장벽 투과(BBBP)

QSAR(회귀)과 **동일한 "기술자→모델" 기계**가 ADMET **분류**에도 그대로 쓰입니다.
여기서는 **혈뇌장벽 투과(BBBP)** 를 ECFP 지문 + 랜덤포레스트로 예측합니다(불균형 데이터 → **ROC-AUC** 로 평가).

In [ ]:
# BBBP(혈뇌장벽 투과) — 실제 공개 데이터. §2의 descriptors()·mfp 재사용.
bbbp=pd.read_csv("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv")
bbbp["mol"]=bbbp["smiles"].apply(Chem.MolFromSmiles)
bbbp=bbbp[bbbp["mol"].notnull()].reset_index(drop=True)   # RDKit 유효 분자만
Xb_fp=np.array([list(mfp.GetFingerprint(m)) for m in bbbp["mol"]])
yb=bbbp["p_np"].values
u,c=np.unique(yb,return_counts=True)
print("BBBP 분자:", len(bbbp), "| 클래스(0=비투과,1=투과):", dict(zip(u.tolist(),c.tolist())))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve

Xtrb,Xteb,ytrb,yteb=train_test_split(Xb_fp,yb,test_size=0.2,random_state=42,stratify=yb)
clf_b=RandomForestClassifier(n_estimators=400,random_state=42,n_jobs=-1,
                             class_weight="balanced").fit(Xtrb,ytrb)
proba=clf_b.predict_proba(Xteb)[:,1]; pred=(proba>=0.5).astype(int)
auc=roc_auc_score(yteb,proba); acc=accuracy_score(yteb,pred)
cvb=cross_val_score(RandomForestClassifier(n_estimators=400,random_state=42,n_jobs=-1,class_weight="balanced"),
                    Xb_fp,yb,cv=StratifiedKFold(5,shuffle=True,random_state=42),scoring="roc_auc")
print(f"BBBP (ECFP+RF) 테스트 ROC-AUC={auc:.3f} | 정확도={acc:.3f}")
print(f"5-fold CV ROC-AUC={cvb.mean():.3f} ± {cvb.std():.3f}  (무작위 0.5)")


In [ ]:
# 시각화: ROC 곡선 + 혼동행렬
fig,ax=plt.subplots(1,2,figsize=(11,4.6))
fpr,tpr,_=roc_curve(yteb,proba)
ax[0].plot(fpr,tpr,color="#C44E52",lw=2,label=f"AUC={auc:.3f}")
ax[0].plot([0,1],[0,1],"k--",lw=1)
ax[0].set_xlabel("위양성률(FPR)"); ax[0].set_ylabel("민감도(TPR)")
ax[0].set_title("BBBP ROC 곡선"); ax[0].legend(loc="lower right")
cm=confusion_matrix(yteb,pred)
im=ax[1].imshow(cm,cmap="Blues")
for (i,j),v in np.ndenumerate(cm):
    ax[1].text(j,i,int(v),ha="center",va="center",
               color="white" if v>cm.max()/2 else "black",fontsize=12)
ax[1].set_xticks([0,1]); ax[1].set_xticklabels(["비투과","투과"])
ax[1].set_yticks([0,1]); ax[1].set_yticklabels(["비투과","투과"])
ax[1].set_xlabel("예측"); ax[1].set_ylabel("실제"); ax[1].set_title("혼동행렬 (BBBP)")
plt.colorbar(im,ax=ax[1],fraction=0.046)
plt.tight_layout(); plt.show()


## 7. 정리

- **표현이 성능을 좌우**: 같은 데이터라도 기술자 vs 지문, 모델에 따라 R²/RMSE가 달라짐(실측 표 참고).
- **검증이 QSAR의 핵심**: CV로 안정성 확인, **y-scrambling**으로 우연 상관 배제, **적용범위(AD)** 밖은 신뢰도↓.
- **해석**: 특징 중요도로 logS를 좌우하는 물성(예: logP·TPSA)을 확인 — 기술자 기반 QSAR의 장점.
- **QSAR = ADMET**: 회귀(용해도)든 분류(혈뇌장벽 투과)든 **동일한 표현→모델 파이프라인**. 불균형 분류는 ROC-AUC로 평가.

> ⚠️ 교육용 데모입니다. 실제 QSAR/ADMET는 scaffold split·외부검증·더 큰 데이터·불확실성 정량이 필요하며, 예측은 실험 검증 전까지 결론이 아닙니다.

**참고문헌**
- Delaney. *ESOL: Estimating Aqueous Solubility Directly from Molecular Structure.* J Chem Inf Comput Sci 44:1000 (2004)
- Martins et al. *A Bayesian Approach to in Silico Blood-Brain Barrier Penetration Modeling.* J Chem Inf Model 52:1686 (2012) — BBBP
- Tropsha. *Best Practices for QSAR Model Development, Validation, and Exploitation.* Mol Inform 29:476 (2010)
- Rogers & Hahn. *Extended-Connectivity Fingerprints.* J Chem Inf Model 50:742 (2010)
- RDKit: Open-source cheminformatics (https://www.rdkit.org)